In [1]:
from IPython.display import display, HTML

display(HTML("""
<style>

/* =========================
   전체 레이아웃
========================= */

div.container{
    width:85% !important;
}

div.cell.code_cell.rendered{
    width:100%;
}

div.input_prompt{
    padding:0;
}

div.prompt{
    min-width:70px;
}

div#toc-wrapper{
    padding-top:120px;
}

table.dataframe{
    font-size:12px;
}

/* =========================
   코드 입력창
========================= */

div.CodeMirror{
    font-family:"마루 부리OTF 중간" !important;
    font-size:12pt !important;
    line-height:1.6;
}

/* =========================
   입력 셀
========================= */

div.input{
    font-family:"마루 부리OTF 중간" !important;
    font-size:12pt !important;
}

/* =========================
   코드 출력
========================= */

div.output{
    font-family:"마루 부리OTF 중간" !important;
    font-size:12pt !important;
}

/* =========================
   Markdown 전체
========================= */

.rendered_html{
    font-family:"마루 부리OTF 중간" !important;
    font-size:18px !important;
    line-height:1.8;
}

/* 제목 */

.rendered_html h1,
.rendered_html h2,
.rendered_html h3,
.rendered_html h4,
.rendered_html h5,
.rendered_html h6{
    font-family:"마루 부리OTF 조금굵은" !important;
}

/* 본문 */

.rendered_html p{
    font-family:"마루 부리OTF 중간" !important;
}

/* 리스트 */

.rendered_html li{
    font-family:"마루 부리OTF 중간" !important;
    padding:5px;
}

/* 인용 */

.rendered_html blockquote{
    font-family:"마루 부리OTF 중간" !important;
}

/* 표 */

.rendered_html table{
    font-family:"마루 부리OTF 중간" !important;
}

/* 코드 블록 */

.rendered_html pre,
.rendered_html code{
    font-family:"Consolas" !important;
    font-size:12pt !important;
}

</style>
"""))


# ※ Quiz : 경주여행과 전주여행에 대해 최빈단어시각화와 유사도분석
- (1) naver open API를 활용하여 블로그에 "경주여행", "전주여행"을 각각 500건씩 검색하여 백업
  (data/quiz/naver.csv)
     * 백업 파일 내용(query, no, title, link, description, total_text(title + ' ' + description)
- (2) naver.csv에서 total_text를 품사태깅(naver_pos.csv)
     * 파일 내용 : query, no, token, pos
- (3) 명사만 추출(naver_pos_nouns.csv)
     * query, token, pos
- (4) 빈도분석 백업(naver_pos_nouns_count.csv)
     * token, 경주빈도, 전주빈도, 빈도합
- (5) 빈도 시각화(워드클라우드, Text.plot)
     * 이미지 저장
- (6) 단어간 거리 분석(Word2Vec), 단어간 연관분석

# 1. 네이버 open API 활용하여 검색 추출
- query, no, title, link, description, total_text(title + ' ' + description)

In [21]:
# .env가져오기
from dotenv import load_dotenv
import os
load_dotenv()

True

In [22]:
# 네이버 개발자 센터에 있는 소스를 가져오기
# 네이버 검색 AIP 예제
# 방법 1 예시
import json
import os
import sys
import requests
import pandas as pd
import urllib.request
from dotenv import load_dotenv
from requests import get

client_id = os.getenv('CLIENT_ID')
client_secret = os.getenv('CLIENT_SECRET')
encText = urllib.parse.quote("경주 여행")
url = f"https://openapi.naver.com/v1/search/blog.json?query={encText}"# JSON 결과

request = urllib.request.Request(url)
request.add_header("X-Naver-Client-Id", client_id)
request.add_header("X-Naver-Client-Secret", client_secret)
response = urllib.request.urlopen(request)
rescode = response.getcode()

if(rescode==200):
    response_body = response.read()
    print(response_body.decode('utf-8')[:200])
else:
    print("Error Code:" + rescode)

{
	"lastBuildDate":"Thu, 03 Sep 2026 17:46:13 +0900",
	"total":2735122,
	"start":1,
	"display":10,
	"items":[
		{
			"title":"3월의 <b>경주여행<\/b>.",
			"link":"https:\/\/lje77777.tistory.com\/7132",
			"


In [23]:
# 문자 -> dict
import json
from html import unescape # description에 있는 &lt;(특수문자)를 <로 변경
import requests
import pandas as pd
import re # 특수문자 제거 !@#$%^&*()-=_+

In [24]:
query = "경주 여행"
start = 1
# url = f"https://openapi.naver.com/v1/search/blog.json?query={query}&display=100&start={start}"
url = "https://openapi.naver.com/v1/search/blog.json"
params = {
    'query':query,
    'display':100,
    'start':start
}
headers = {
    "X-Naver-Client-Id":client_id,
    "X-Naver-Client-Secret":client_secret
}
response = requests.get(url, headers=headers, params=params)

items = json.loads(response.text)['items']
itmes = response.json()['items']
items[4]

{'title': '<b>경주여행</b>/야경이 이쁜 안압지&amp;첨성대【16년3월31일】',
 'link': 'https://skdywjd25.tistory.com/4030',
 'description': '어울린다 <b>경주 여행</b>에서 빠질수 없는 관광지 중 하나인 첨성대.. 첨성대는 신라시대에 별을 관측하기... ^^ㅋ 최근 <b>경주</b>를 <b>여행</b>하는 이들이 꼭 빼놓지 않는 <b>여행</b>코스가 있다. 다름 아닌 <b>경주</b>의 야경을 둘러보는 것.... ',
 'bloggername': '산행과여행 사진으로 말한다',
 'bloggerlink': 'https://skdywjd25.tistory.com/',
 'postdate': '20160402'}

In [27]:
# title과 description의 <b>제거, html의 특수문자 제거, 일반특수 제거
item = items[4]
title = item['title'].replace('<b>', ' ').replace('</b>', ' ')
title = unescape(title)
title = re.sub(r'[^a-zA-Z0-9가-힣]', ' ', title)
description = item['description'].replace('<b>', ' ').replace('</b>', ' ')
description = unescape(description)
description = re.sub(r'[^a-zA-Z0-9가-힣]', '', description)
totletext = title + ' ' + description
totletext

' 경주여행  야경이 이쁜 안압지 첨성대 16년3월31일  어울린다경주여행에서빠질수없는관광지중하나인첨성대첨성대는신라시대에별을관측하기최근경주를여행하는이들이꼭빼놓지않는여행코스가있다다름아닌경주의야경을둘러보는것'

In [26]:
# re 정규표현식을 이용해서 특수문자 제거
title = '[여행] ## & ktx 타고 짱 ㅋㅋ ㅠㅠ'
re.sub(r'[^a-zA-Z0-9가-힣]', ' ', title)

' 여행       ktx 타고 짱      '

In [ ]:
# 네이버 API 정보 및 검색 정보
from dotenv import load_dotenv
import os
load_dotenv()
client_id = os.getenv('CLIENT_ID')
client_secret = os.getenv('CLIENT_SECRET')

queries = ['경주 여행', '전주 여행']
max_start = 5

In [ ]:
def get_search_item_return(query, start):
    'query와 start로 naver 블로그 검색한 결과 title, link, descript, total_text를 dict list'
    pass

In [ ]:
for query in queries:
    for start in range(1, max_start+1):
        get_search_item_return(query, start)